# Análisis exploratorio del Spotify Tracks Dataset

Requiere `data/raw/dataset.csv`. La limpieza vive en `src/recommender/data.py`, así que este notebook y la API usan el mismo código.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

from recommender.config import MODEL_FEATURES, SCALED_FEATURES
from recommender.data import clean, load_raw
from recommender.genres import GENRE_TO_MACRO, UNKNOWN_MACRO_GENRE

RAW_PATH = Path("../data/raw/dataset.csv")
CLEAN_PATH = Path("../data/processed/tracks_clean.csv")
BAR_COLOR = "#3B6FB6"

raw = load_raw(RAW_PATH)
raw.head()

In [ ]:
song_key = [
    raw["track_name"].str.strip().str.casefold(),
    raw["artists"].str.strip().str.casefold()
]

resumen = pd.Series(
    {
        "filas": len(raw),
        "track_id únicos": raw["track_id"].nunique(),
        "canciones únicas (nombre + artistas)": raw.groupby(song_key).ngroups,
        "géneros": raw["track_genre"].nunique(),
        "filas con nulos": raw.isna().any(axis=1).sum(),
    }
)

resumen

In [ ]:
raw[raw.isna().any(axis=1)]

## 2. Géneros sin macro-género

Ambas listas deben quedar vacías. Si no, hay que corregir `src/recommender/genres.py`.

In [ ]:
dataset_genres = set(raw["track_genre"].dropna())

print("En el dataset, sin mapear:", sorted(dataset_genres - set(GENRE_TO_MACRO)))
print("Mapeados, no están en el dataset:", sorted(set(GENRE_TO_MACRO) - dataset_genres))

## 3. Distribución de las características

In [ ]:
raw[MODEL_FEATURES].describe().T

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 8), constrained_layout=True)

for ax, feature in zip(axes.flat, SCALED_FEATURES):
    ax.hist(raw[feature], bins=40, color=BAR_COLOR)
    ax.set_title(feature)
    ax.spines[["top", "right"]].set_visible(False)

fig.supylabel("Canciones")
plt.show()

## 4. Correlación entre características

Dos variables muy correlacionadas pesan casi doble en la similitud coseno.

In [ ]:
corr = raw[SCALED_FEATURES].corr()

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
image = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)

for row, col in [(r, c) for r in range(len(corr)) for c in range(len(corr))]:
    value = corr.iloc[row, col]
    ax.text(
        col,
        row,
        f"{value:.2f}",
        ha="center",
        va="center",
        fontsize=8,
        color="white" if abs(value) > 0.6 else "black"
    )

fig.colorbar(image, ax=ax, label="Correlación", shrink=0.8)
plt.show()

## 5. Popularidad

Sirve de referencia para medir el sesgo de popularidad de las recomendaciones.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.hist(raw["popularity"], bins=50, color=BAR_COLOR)
ax.set_xlabel("Popularidad")
ax.set_ylabel("Canciones")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print("Canciones con popularidad 0:", (raw["popularity"] == 0).sum())

## 6. Valores sospechosos

Un tempo de 0 indica que no se detectó ritmo. Suele ser ruido blanco o audio para dormir, así que `clean` quita esas filas y conserva la columna `tempo`.

In [ ]:
sin_tempo = raw[raw["tempo"] == 0]

print("Filas con tempo 0:", len(sin_tempo))
print(sin_tempo["track_genre"].value_counts().head(10))

## 7. Limpieza

In [ ]:
tracks = clean(raw)

print(f"Antes: {len(raw):,} filas")
print(f"Después: {len(tracks):,} filas")
print(f"Quitadas: {len(raw) - len(tracks):,} ({1 - len(tracks) / len(raw):.1%})")
print("Sin macro-género:", (tracks["macro_genre"] == UNKNOWN_MACRO_GENRE).sum())
print("track_id repetidos:", tracks["track_id"].duplicated().sum())

In [ ]:
counts = tracks["macro_genre"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
bars = ax.barh(counts.index, counts.values, color=BAR_COLOR)
ax.bar_label(bars, fmt="{:,.0f}", padding=4)
ax.set_title("Canciones por macro-género después de limpiar", loc="left")
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.xaxis.set_visible(False)
plt.show()

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
tracks.to_csv(CLEAN_PATH, index=False)

print("Guardado en", CLEAN_PATH)